In [ ]:
#Q1.  In your own words, what is a decorator? Write a one-paragraph explanation without
looking at your notes.

In [ ]:
#Q2.  Write a simple decorator called my_decorator that prints 'Function is starting' before
#and 'Function is starting' before and 'Function is done' after any function it wraps. Apply it to a function greet() that prints

In [ ]:
def my_decorator(func):
    def wrapper():
        print("Function is starting")
        func()
        print("Function is done")
    return wrapper

@my_decorator
def greet():
    print("Hello!")

greet()

In [ ]:
 # What is the purpose of *args and **kwargs in the wrapper function inside a decorator?
#Why is it important to include them?

In [ ]:
def my_decorator(func):
    def wrapper(*args, **kwargs):
        print("Before function")
        result = func(*args, **kwargs)
        print("After function")
        return result
    return wrapper

@my_decorator
def add(a, b):
    return a + b

print(add(10, 20))

In [ ]:
#Q4.  What does @functools.wraps(func) do? Write an example showing what happens to __name__ with and without it

In [ ]:
def my_decorator(func):
    def wrapper():
        print("Running...")
        func()
    return wrapper

@my_decorator
def greet():
    print("Hello")

print(greet.__name__)

#with wraps
from functools import wraps

def my_decorator(func):
    @wraps(func)
    def wrapper():
        print("Running...")
        func()
    return wrapper

@my_decorator
def greet():
    print("Hello")

print(greet.__name__)

In [ ]:
#Q1.  Write a decorator called validate_positive that checks all positional arguments passed
#to a function. If any argument is negative, print an error message and return None without

In [ ]:
def validate_positive(func):
    def wrapper(*args):
        for value in args:
            if value < 0:
                print("Error: Negative values are not allowed.")
                return None
        return func(*args)
    return wrapper

@validate_positive
def multiply(a, b):
    return a * b

print(multiply(5, 6))
print(multiply(5, -6))

In [ ]:
#Q2.  Create a decorator factory called repeat(n) that runs the decorated function exactly n
times. Then stack it with a logger decorator and apply both to a function. Trace the exact order in which the wrappers execute.


In [ ]:
def logger(func):
    def wrapper(*args, **kwargs):
        print("Logger: Before function")
        result = func(*args, **kwargs)
        print("Logger: After function")
        return result
    return wrapper

def repeat(n):
    def decorator(func):
        def wrapper(*args, **kwargs):
            for i in range(n):
                print(f"Repeat {i+1}")
                func(*args, **kwargs)
        return wrapper
    return decorator

@logger
@repeat(3)
def greet():
    print("Hello!")

greet()

In [ ]:
#Q3.  Write a decorator called count_calls that tracks how many times a function has been called and prints the count each time. Hint: you will need to store state — use a mutable object (like a list or a dictionary attribute on the wrapper).


In [ ]:
def count_calls(func):
    calls = {"count": 0}

    def wrapper(*args, **kwargs):
        calls["count"] += 1
        print(f"Function called {calls['count']} time(s)")
        return func(*args, **kwargs)

    return wrapper

@count_calls
def greet():
    print("Hello!")

greet()
greet()
greet()

In [ ]:
#Q4.  Explain the difference between these two usages and what error (if any) the wrong one produces:     Option A:  @my_decorator     Option B:  @my_decorator()     When is each form correct?

In [ ]:
@my_decorator
def greet():
    print("Hello")
#This is correct when my_decorator directly accepts a function as its argument. equivalent to
greet = my_decorator(greet)

@my_decorator()
def greet():
    print("Hello")
#This is correct only when my_decorator is a decorator factory, meaning it first returns another decorator.

#example
def my_decorator():
    def decorator(func):
        def wrapper():
            print("Before")
            func()
            print("After")
        return wrapper
    return decorator

In [ ]:
#Q1.  Build a complete caching decorator called memoize that stores the results of previous calls in a dictionary (keyed by the arguments). If the same arguments are passed again, return the cached result without re-running the function. Test it on a recursive Fibonacci function and show how many function calls are saved.


In [ ]:
from functools import wraps

def memoize(func):
    cache = {}

    @wraps(func)
    def wrapper(*args):
        if args in cache:
            print(f"Cache hit for {args}")
            return cache[args]

        print(f"Calculating for {args}")
        result = func(*args)
        cache[args] = result
        return result

    return wrapper


calls = {"count": 0}

@memoize
def fibonacci(n):
    calls["count"] += 1

    if n <= 1:
        return n

    return fibonacci(n - 1) + fibonacci(n - 2)


print("Answer:", fibonacci(10))
print("Actual function executions:", calls["count"])

In [ ]:
#Q2.  Write a decorator factory called rate_limit(max_calls, period_seconds) that allows a function to be called at most max_calls times within any period_seconds window. If the limit is exceeded, raise a RuntimeError with a clear message.Use time.time() to track thewindow.


In [ ]:
import time
from functools import wraps

def rate_limit(max_calls, period_seconds):

    def decorator(func):
        call_times = []

        @wraps(func)
        def wrapper(*args, **kwargs):

            current = time.time()

            # Remove old timestamps
            while call_times and current - call_times[0] > period_seconds:
                call_times.pop(0)

            if len(call_times) >= max_calls:
                raise RuntimeError(
                    f"Rate limit exceeded. Only {max_calls} calls allowed every {period_seconds} seconds."
                )

            call_times.append(current)

            return func(*args, **kwargs)

        return wrapper

    return decorator


@rate_limit(3, 5)
def greet():
    print("Hello")


greet()
greet()
greet()

# Fourth call within 5 seconds
greet()

In [ ]:
#Q3.  Implement a class-based decorator (using __init__ and __call__) called RetryOnException that retries the decorated function up to n times if it raises a specific exception type. The exception type and number of retries should be passed to the decorator factory. Demonstrate it on a function that randomly fails.


In [ ]:
import random

class RetryOnException:

    def __init__(self, exception_type, retries):
        self.exception_type = exception_type
        self.retries = retries

    def __call__(self, func):

        def wrapper(*args, **kwargs):

            for attempt in range(1, self.retries + 1):

                try:
                    print(f"Attempt {attempt}")
                    return func(*args, **kwargs)

                except self.exception_type as e:

                    print("Failed:", e)

                    if attempt == self.retries:
                        print("Maximum retries reached.")
                        raise

                    print("Retrying...\n")

        return wrapper


@RetryOnException(ValueError, 3)
def unreliable():

    if random.random() < 0.7:
        raise ValueError("Random failure")

    print("Success!")


unreliable()

In [ ]:
#Q4.  Design a decorator pipeline: write three separate decorators — @authenticate(role), @timer, and @logger — and apply all three to a single function. Then write a detailed explanation of the exact order in which execution flows through the three wrappers when the function i

In [ ]:
#step1
import time
from functools import wraps

def logger(func):

    @wraps(func)
    def wrapper(*args, **kwargs):
        print("Logger: Function started")

        result = func(*args, **kwargs)

        print("Logger: Function finished")

        return result

    return wrapper

#step2
def timer(func):

    @wraps(func)
    def wrapper(*args, **kwargs):

        start = time.time()

        result = func(*args, **kwargs)

        end = time.time()

        print(f"Execution Time: {end-start:.5f} seconds")

        return result

    return wrapper

#3
def authenticate(role):

    def decorator(func):

        @wraps(func)
        def wrapper(user_role, *args, **kwargs):

            if user_role != role:
                print("Access Denied")
                return

            print("Authentication Successful")

            return func(user_role, *args, **kwargs)

        return wrapper

    return decorator

@logger
@timer
@authenticate("admin")
def delete_database(user_role):

    print("Database Deleted")


delete_database("admin")

In [4]:
def electricity(rate_per_unit):
    def inner(units):
        return units*rate_per_unit
    return inner

x=electricity(20)
print(x(30))

print(x.__closure__)
print(x.__closure__[0])
print(x.__closure__[0].cell_contents)


600
(<cell at 0x000001B572B4D930: int object at 0x00007FFE50F08608>,)
<cell at 0x000001B572B4D930: int object at 0x00007FFE50F08608>
20
